In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("natezhang123/social-anxiety-dataset")

print("Path to dataset files:", path)

100%|██████████| 403k/403k [00:00<00:00, 41.2MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/natezhang123/social-anxiety-dataset/versions/2


In [ ]:
# 📂 Load the dataset
df = pd.read_csv("/content/enhanced_anxiety_dataset.csv")

# ===============================================
# 🧹 Data Preprocessing
# ===============================================

# Convert continuous anxiety level (1–10) into binary:
# 1 → Anxious, 0 → Not Anxious
df["Anxious"] = (df["Anxiety Level (1-10)"] >= 5).astype(int)

# Separate features (X) and target (y)
X = df.drop(columns=["Anxiety Level (1-10)", "Anxious"])
y = df["Anxious"]

# Convert categorical features into numeric (One-Hot Encoding)
X = pd.get_dummies(X, drop_first=True)

# Normalize numerical data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================================
# 🔀 Split the data
# ===============================================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ===============================================
# 🤖 Train the model
# ===============================================
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# ===============================================
# 📊 Evaluate the model
# ===============================================
y_pred = model.predict(X_test)

print("✅ Model Evaluation Results")
print("-----------------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%\n")
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=["Not Anxious", "Anxious"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ===============================================
# 🧪 Predict new patient (example)
# ===============================================
new_patient = pd.DataFrame([{
    "Age": 24,
    "Gender": "Female",
    "Occupation": "Student",
    "Sleep Hours": 5.5,
    "Physical Activity (hrs/week)": 1.0,
    "Caffeine Intake (mg/day)": 200,
    "Alcohol Consumption (drinks/week)": 2,
    "Smoking": "No",
    "Family History of Anxiety": "Yes",
    "Stress Level (1-10)": 8,
    "Heart Rate (bpm)": 95,
    "Breathing Rate (breaths/min)": 22,
    "Sweating Level (1-5)": 4,
    "Dizziness": "Yes",
    "Medication": "No",
    "Therapy Sessions (per month)": 0,
    "Recent Major Life Event": "Yes",
    "Diet Quality (1-10)": 3
}])

# One-hot encode and align columns with training data
new_patient = pd.get_dummies(new_patient)
new_patient = new_patient.reindex(columns=X.columns, fill_value=0)

# Scale
new_patient_scaled = scaler.transform(new_patient)

# Predict
prediction = model.predict(new_patient_scaled)[0]

if prediction == 1:
    print("\n🧠 Prediction: The patient is likely ANXIOUS.")
else:
    print("\n🙂 Prediction: The patient is NOT anxious.")


In [ ]:
import joblib
joblib.dump(model, "anxiety_detection_model.pkl")
joblib.dump(scaler, "anxiety_detection_scaler.pkl")
joblib.dump(X.columns, "anxiety_detection_features.pkl")

['anxiety_detection_features.pkl']

In [ ]:
model = joblib.load("anxiety_detection_model.pkl")


In [ ]:
import gradio as gr
import pandas as pd
import joblib
import numpy as np

# -----------------------------
# Load your trained model, scaler, and feature columns
# -----------------------------
model = joblib.load("/content/anxiety_detection_model.pkl")
scaler = joblib.load("/content/anxiety_detection_scaler.pkl")
features_columns = joblib.load("/content/anxiety_detection_features.pkl")

# -----------------------------
# Prediction Function
# -----------------------------
def predict_anxiety(age, gender, occupation, sleep_hours, physical_activity_hrs_week,
                    caffeine_intake, alcohol_consumption, smoking, family_history_anxiety,
                    stress_level, heart_rate, breathing_rate, sweating_level,
                    dizziness, medication, therapy_sessions, recent_major_life_event, diet_quality):

    # Create a DataFrame from the inputs, matching original feature names
    new_patient_data = {
        "Age": age,
        "Gender": gender,
        "Occupation": occupation,
        "Sleep Hours": sleep_hours,
        "Physical Activity (hrs/week)": physical_activity_hrs_week,
        "Caffeine Intake (mg/day)": caffeine_intake,
        "Alcohol Consumption (drinks/week)": alcohol_consumption,
        "Smoking": smoking,
        "Family History of Anxiety": family_history_anxiety,
        "Stress Level (1-10)": stress_level,
        "Heart Rate (bpm)": heart_rate,
        "Breathing Rate (breaths/min)": breathing_rate,
        "Sweating Level (1-5)": sweating_level,
        "Dizziness": dizziness,
        "Medication": medication,
        "Therapy Sessions (per month)": therapy_sessions,
        "Recent Major Life Event": recent_major_life_event,
        "Diet Quality (1-10)": diet_quality
    }

    new_patient_df = pd.DataFrame([new_patient_data])

    # One-hot encode categorical features
    new_patient_df = pd.get_dummies(new_patient_df, drop_first=True)

    # Reindex to align columns with training data, filling missing with 0
    # This is crucial for consistent feature order and handling unseen categories
    new_patient_aligned = new_patient_df.reindex(columns=features_columns, fill_value=0)

    # Scale numerical features using the loaded scaler
    new_patient_scaled = scaler.transform(new_patient_aligned)

    prediction = model.predict(new_patient_scaled)[0]

    if prediction == 1:
        return "🧠 Predicted: The patient is likely ANXIOUS."
    else:
        return "🙂 Predicted: The patient is NOT anxious."

# -----------------------------
# Build Gradio Interface
# -----------------------------
interface = gr.Interface(
    fn=predict_anxiety,
    inputs=[
        gr.Number(label="Age", value=24),
        gr.Radio(["Male", "Female", "Other"], label="Gender", value="Female"),
        gr.Dropdown(["Student", "Artist", "Engineer", "Doctor", "Teacher", "Other"], label="Occupation", value="Student"),
        gr.Number(label="Sleep Hours", value=5.5),
        gr.Number(label="Physical Activity (hrs/week)", value=1.0),
        gr.Number(label="Caffeine Intake (mg/day)", value=200),
        gr.Number(label="Alcohol Consumption (drinks/week)", value=2),
        gr.Radio(["Yes", "No"], label="Smoking", value="No"),
        gr.Radio(["Yes", "No"], label="Family History of Anxiety", value="Yes"),
        gr.Slider(1, 10, step=1, label="Stress Level (1-10)", value=8),
        gr.Number(label="Heart Rate (bpm)", value=95),
        gr.Number(label="Breathing Rate (breaths/min)", value=22),
        gr.Slider(1, 5, step=1, label="Sweating Level (1-5)", value=4),
        gr.Radio(["Yes", "No"], label="Dizziness", value="Yes"),
        gr.Radio(["Yes", "No"], label="Medication", value="No"),
        gr.Number(label="Therapy Sessions (per month)", value=0),
        gr.Radio(["Yes", "No"], label="Recent Major Life Event", value="Yes"),
        gr.Slider(1, 10, step=1, label="Diet Quality (1-10)", value=3)
    ],
    outputs=gr.Textbox(label="Anxiety Prediction"),
    title="Social Anxiety Predictor",
    description="Enter your lifestyle & behavioral features to predict anxiety level."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14e1fcdce2e135f596.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
